# Extract images of individual samples from whole slide image

Per run of technologies such as Xenium In Situ or Merscope, multiple tissue sections can be measured. While the spatial transcriptomics measurements of the tissue sections are often done per tissue sections, scans of subsequent H&E stainings often contain all tissue sections at once. This makes it necessary to extract the individual H&E scans from the whole slide image before registering them to the spatial transcriptomics datasets. Here, we present a `InSituPy`-based approach to achieve this.

In [1]:
## The following code ensures that all functions and init files are reloaded before executions.
%load_ext autoreload
%autoreload 2

In [2]:
# import packages
from insitupy import InSituData, CACHE
from insitupy.dataclasses import ImageData
from insitupy import InSituExperiment

## Load images and create an `InSituData` object

In [13]:
# Initialize an empty InSituData object
xd = InSituData()

# add image
he_path = CACHE / "demo_datasets/hbreastcancer/unregistered_images/slide_id__hbreastcancer__HE__histo.ome.tif"
xd.images.add_image(image=he_path, name="HE")

In [14]:
xd

InSituData
Method:		not specified
Slide ID:	None
Sample ID:	None
Path:		None

    ➤ images
       HE:	(24241, 30786, 3)

## Visualize the data to select the individual samples

In [15]:
xd.show()

INFO: Added 1 new regions to key 'TestKey'
INFO: Added 1 new regions to existing key 'TestKey'
INFO: Added 1 new regions to existing key 'TestKey'


### Add regions to image

To add regions, one can use the "Add geometries" widget on the right side of the napari window:
1. Select Type = "Regions"
2. Add a Key value and a Class name
3. Press Add geometry layer
4. Add one rectangle per region layer. <b>!!! Important</b>: Per region layer only one shape is allowed.
5. Repeat this for all regions.

<left><img src="../../demo_data/demo_screenshots/select_sample_region.jpg" width="800"/></left>

### Import the regions from the napari viewer into the `InSituData` object

Click "Sync Geometries" on the bottom right of the window or run the function below.

In [ ]:
# from insitupy.interactive import sync_geometries
# sync_geometries()

In [16]:
xd

InSituData
Method:		not specified
Slide ID:	None
Sample ID:	None
Path:		None

    ➤ images
       HE:	(24241, 30786, 3)
    ➤ regions
       TestKey:	3 regions, 3 classes ('Region 1', 'Region 2', 'Region 3') 

## Create an `InSituExperiment` object using the annotated regions

In [24]:
exp = InSituExperiment.from_regions(
    data=xd, region_key="TestKey"
)

Region 1
Region 2
Region 3


In [25]:
exp

InSituExperiment with 3 samples:
           uid  CITAR slide_id sample_id region_key region_name
0     11960871  -+--+     None      None    TestKey    Region 1
1     ccb01730  -+--+     None      None    TestKey    Region 2
2     dd68d331  -+--+     None      None    TestKey    Region 3

## Save the individual images

In [26]:
for m, d in exp.iterdata():
    d.images.save(output_folder=f"out/{m['region_name']}", as_zarr=False)

After saving the individual images, one can now continue by registering the images to the spatial transcriptomics data as shown in notebook `01_InSituPy_demo_register_images.ipynb`.